In [ ]:
# prompt: mount google drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install scanpy

In [ ]:
import scanpy as sc

# load the data (harmony integrated)

In [ ]:
datapath='/content/drive/MyDrive/Colab Notebooks/scGPT/pancreatic/data/TF-PerturbSeq/small_scale/merged_harmony_integrated_ESC.h5ad'

adata_full=sc.read_h5ad(datapath)




In [ ]:
adata_full

In [ ]:
adata_full.var

In [ ]:
adata_full.obs

In [ ]:
adata_full.layers['logcounts'][:10,:20].toarray()

In [ ]:
adata_full.X[:10,:20].toarray()

In [ ]:
adata_full

# compare the distribution of expression for cells with or without certain perturbation, not based on the highest assignment

In [ ]:
sc.pp.neighbors(adata_full, use_rep='umap_rna')
sc.tl.umap(adata_full)

In [ ]:
perturbed_gene='NANOG'
adata_full.obs['has_target_gene']=0

adata_full.obs.iloc[adata_full.obs.genotype.str.startswith(perturbed_gene),adata_full.obs.columns.get_loc('has_target_gene')]=1
sum(adata_full.obs.has_target_gene)

In [ ]:
adata_full.obs.loc[adata_full.obs.genotype.str.startswith(perturbed_gene),]

In [ ]:
sc.pl.umap(adata_full,color=['harmony_snn_res.0.3','harmony_snn_res.1',],)

In [ ]:
sc.pl.umap(adata_full,color=['percent.mt','percent.ribo',],)

In [ ]:
adata_full.var.loc[adata_full.var.gene_ids.str.startswith('NANOG'),]

In [ ]:
sc.pl.umap(adata_full,color=['POU5F1','NANOG'],)

In [ ]:
sc.pl.umap(adata_full,color=[perturbed_gene,'has_target_gene',],size=5, legend_loc='on data')

In [ ]:
sc.pl.violin(adata_full, keys='NANOG', groupby='harmony_snn_res.0.3', rotation=90)


In [ ]:
sc.pl.violin(adata_full, keys='NANOG', groupby='harmony_snn_res.0.3',layer='logcounts', rotation=90)


In [ ]:
# prompt: plot the cumulative distribution for the NANOG expression of cells in 2 groups, depending on whether their has_target_gene column in adata.obs is 0 or 1.

import matplotlib.pyplot as plt
import seaborn as sns

# Extract NANOG expression for cells in group 0 and group 1
nanog_expression_group0 = adata_full[adata_full.obs['has_target_gene'] == 0, perturbed_gene].X.toarray().flatten()
nanog_expression_group1 = adata_full[adata_full.obs['has_target_gene'] == 1, perturbed_gene].X.toarray().flatten()

nanog_expression_group0 = adata_full[adata_full.obs['has_target_gene'] == 0, perturbed_gene].layers['logcounts'].toarray().flatten()
nanog_expression_group1 = adata_full[adata_full.obs['has_target_gene'] == 1, perturbed_gene].layers['logcounts'].toarray().flatten()


# Plot cumulative distribution
plt.figure(figsize=(8, 6))
sns.ecdfplot(data=nanog_expression_group0, label='has_target_gene == 0')
sns.ecdfplot(data=nanog_expression_group1, label='has_target_gene == 1')
plt.title(f'Cumulative Distribution of {perturbed_gene} Expression')
plt.xlabel(f'{perturbed_gene} Expression')
plt.ylabel('Cumulative Probability')
plt.legend()
plt.grid(True)
plt.ylim(0.0, 1)
plt.show()


In [ ]:
nanog_expression_group1.shape

In [ ]:
# prompt: perform k-s test for nanog_expression_group0 and nanog_expression_group1

from scipy.stats import ks_2samp

# Perform the K-S test
ks_statistic, p_value = ks_2samp(nanog_expression_group0, nanog_expression_group1)

# Print the results
print(f"K-S Statistic: {ks_statistic}")
print(f"P-value: {p_value}")


# save to h5ad file

In [ ]:
adata.write_h5ad('ESC_TF_perturbseq_processed.h5ad')


In [ ]:
adata

In [ ]:
adata.obs

In [ ]:
adata.obs.target_gene.value_counts()

In [ ]:
!cp ESC_TF_perturbseq_processed.h5ad '/content/drive/MyDrive/Colab Notebooks/scGPT/pancreatic/data/TF-PerturbSeq/'

In [ ]:
import scanpy as sc
adata_r=sc.read_h5ad('ESC_TF_perturbseq_processed.h5ad')

In [ ]:
adata_r.obs

# load the data (reprocessed)

In [ ]:
datapath='/content/drive/MyDrive/Colab Notebooks/scGPT/pancreatic/data/TF-PerturbSeq/ESC_TF_perturbseq_processed.h5ad'

adata_full=sc.read_h5ad(datapath)
datapath_g='/content/drive/MyDrive/Colab Notebooks/scGPT/pancreatic/data/TF-PerturbSeq/ESC_TF_perturbseq_processed_guides.h5ad'


adata_guides=sc.read_h5ad(datapath_g)

In [ ]:
adata_full

In [ ]:
adata_guides

In [ ]:
adata_guides.var

In [ ]:
guide_threshold=3
guide_matrix_bin=(adata_guides.X > guide_threshold ) * 1
import numpy as np

guide_rs=np.sum(guide_matrix_bin,axis=1)#.flatten()

guide_rs.shape

In [ ]:
guide_rs=np.array(guide_rs)[:,0]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.histplot(guide_rs, bins=range(0, int(np.max(guide_rs)) + 2), kde=False)
plt.title('Histogram of guide_rs')
plt.xlabel('Number of Guides > Threshold')
plt.ylabel('Number of Cells')
plt.xticks(np.arange(0, int(np.max(guide_rs)) + 2, 1))
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
sum(guide_rs==1)

In [ ]:
adata_guides.X[:10,:10].toarray()

In [ ]:

import numpy as np
gfxcp=adata_guides.X.toarray()
# Get the column index of the highest value for each row of guidef.X
highest_value_indices = np.argmax(gfxcp, axis=1)

# Print the indices
highest_value_indices

highest_values = np.max(gfxcp,axis=1)

# prompt: generate a [i, highest_value_indices[i]] duplex for each position i in highest_value_indices, and set the corresponding positions in gfxcp to zero

# Generate a duplex for each position i in highest_value_indices
# and set the corresponding positions in gfxcp to zero
duplex_indices = [(i, highest_value_indices[i]) for i in range(len(highest_value_indices))]
for i, j in duplex_indices:
    gfxcp[i, j] = 0

# Get the column index of the highest value for each row of guidef.X
second_highest_value_indices = np.argmax(gfxcp, axis=1)
second_highest_values = np.max(gfxcp,axis=1)



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(highest_values, bins=range(int(highest_values.min()), int(highest_values.max()) + 2), kde=False, color='skyblue', label='Highest Values', alpha=0.7)
sns.histplot(second_highest_values, bins=range(int(second_highest_values.min()), int(second_highest_values.max()) + 2), kde=False, color='salmon', label='Second Highest Values', alpha=0.7)

plt.title('Histogram of Highest and Second Highest Guide Counts')
plt.xlabel('Guide Count')
plt.ylabel('Number of Cells')
plt.xscale('log')
plt.yscale('log')
plt.legend()
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.scatterplot(x=highest_values, y=second_highest_values, alpha=0.5)
plt.xscale('log')
plt.yscale('log')
plt.title('Scatter Plot of Highest vs. Second Highest Guide Counts (Log Scale)')
plt.xlabel('Highest Guide Count (log scale)')
plt.ylabel('Second Highest Guide Count (log scale)')
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.show()

In [ ]:
adata_guides.obs

In [ ]:
adata_guides.var

In [ ]:

import numpy as np

g_obs_copy=adata_guides.obs.copy()


g_obs_copy["gene_ids"] = "unassigned"
g_obs_copy["target_gene"]=None
g_obs_copy["highest_guide_count"] = 0
# Iterate through each row of guidef.X
for i in range(g_obs_copy.shape[0]):
    if i % 10000 == 0:
        print(f"processing {i} cells...")

    h_value = highest_values[i]
    h_index=highest_value_indices[i]
    sh_value = second_highest_values[i]
    #sh_index = second_highest_value_indices[i]
    g_obs_copy.iloc[i, g_obs_copy.columns.get_loc("highest_guide_count")] = h_value
    if h_value == 0:
        g_obs_copy.iloc[i, g_obs_copy.columns.get_loc("gene_ids")] = "unassigned"
        g_obs_copy.iloc[i, g_obs_copy.columns.get_loc("target_gene")] = "unassigned"
    elif h_value > 2 * sh_value:
        g_obs_copy.iloc[i, g_obs_copy.columns.get_loc("gene_ids")] = adata_guides.var.iloc[h_index, adata_guides.var.columns.get_loc("gene_ids")]
        g_obs_copy.iloc[i, g_obs_copy.columns.get_loc("target_gene")] = adata_guides.var.iloc[h_index, adata_guides.var.columns.get_loc("genes_2")]
    else:
        g_obs_copy.iloc[i, g_obs_copy.columns.get_loc("gene_ids")] = "ambiguous"
        g_obs_copy.iloc[i, g_obs_copy.columns.get_loc("target_gene")] = "ambiguous"

g_obs_copy



In [ ]:

g_obs_copy.gene_ids.value_counts()

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', None)
g_obs_copy.gene_ids.value_counts()

In [ ]:
g_obs_copy.target_gene.value_counts()

# redo the clustering analysis

In [ ]:
pd.set_option('display.max_rows', 50)

In [ ]:
adata_full.var

In [ ]:
sum(adata_full.var.highly_variable)

In [ ]:
sc.pp.highly_variable_genes(adata_full,n_top_genes=5000)


In [ ]:
sc.tl.pca(adata_full,n_comps=100)

sc.pl.pca_variance_ratio(adata_full, n_pcs=30, log=True)

In [ ]:
adata_full.obsm['X_pca'].shape

In [ ]:



sc.pp.neighbors(adata_full)

sc.tl.umap(adata_full)


In [ ]:
adata_full

In [ ]:


with plt.rc_context({"figure.figsize": (16, 12)}):
    sc.pl.umap(adata_full,color='target_gene',size=30)

In [ ]:
adata_full_2 = adata_full[~adata_full.obs.target_gene.isin(['ambiguous','non','unassigned'])]

with plt.rc_context({"figure.figsize": (8, 6)}):
    sc.pl.umap(adata_full_2,color='target_gene',size=30, frameon=False)

In [ ]:
perturbed_gene='EZH2'
neg_ctrl_group='ambiguous'


adata_full.obs['has_target_gene']=0

#adata.obs.iloc[cellular_index,adata.obs.columns.get_loc('has_target_gene')]=1
adata_full.obs.loc[adata_full.obs.target_gene==perturbed_gene,'has_target_gene']=1
#adata.obs.loc[adata.obs.target_gene==neg_ctrl_group,'has_target_gene']=2
adata_full.obs.loc[(~adata_full.obs.target_gene.isin([perturbed_gene,"unassigned"])),'has_target_gene']=2
adata_full.obs['has_target_gene'] = adata_full.obs['has_target_gene'].astype('category')

sc.pl.umap(adata_full,color=[perturbed_gene,'has_target_gene',],size=10)

In [ ]:
adata_selected =adata_full[~adata_full.obs.target_gene.isin(['unassigned','ambiguous']),:]
sc.pl.umap(adata_selected,color=[perturbed_gene,'has_target_gene',],size=10)

In [ ]:

perturbed_gene='EZH2'
adata_selected.obs['has_target_gene']=0

#adata.obs.iloc[cellular_index,adata.obs.columns.get_loc('has_target_gene')]=1
adata_selected.obs.loc[adata_selected.obs.target_gene==perturbed_gene,'has_target_gene']=1

import numpy as np

random_numbers = np.random.rand(adata_selected.n_obs)
print(random_numbers)

sc.pl.umap( adata_selected[(random_numbers < 0.1) | (adata_selected.obs.has_target_gene == 1),],color=[perturbed_gene,'has_target_gene',],size=10,color_map='Blues')

In [ ]:
adata_selected.obs.target_gene.value_counts()

In [ ]:

for perturbed_gene in adata_selected.obs.target_gene.unique():
  adata_selected.obs['has_target_gene']=0

  #adata.obs.iloc[cellular_index,adata.obs.columns.get_loc('has_target_gene')]=1
  adata_selected.obs.loc[adata_selected.obs.target_gene==perturbed_gene,'has_target_gene']=1

  import numpy as np

  random_numbers = np.random.rand(adata_selected.n_obs)
  print(random_numbers)

  adata_sel2=adata_selected[(random_numbers < 0.1) | (adata_selected.obs.has_target_gene == 1),]
  sc.pl.umap( adata_sel2,color=['has_target_gene',],size=20,color_map='Blues',title=f'pert: {perturbed_gene}')
  if perturbed_gene in adata_selected.var_names:
    sc.pl.umap( adata_sel2,color=[perturbed_gene,],size=10,color_map='Blues')

In [ ]:
with plt.rc_context({"figure.figsize": (16, 12)}):
    fig = sc.pl.umap(adata_full,color='target_gene',size=30,show=False,return_fig=True)
    ax = fig.axes[0]
    ax.set_xticks([-5,0, 5, 10])
    #ay = fig.axes[1]
    ax.set_yticks([ 5, 7])
    plt.show()

In [ ]:
with plt.rc_context({"figure.figsize": (4, 3)}):
    fig = sc.pl.umap(adata_full,color='target_gene',size=30,show=False,return_fig=True)
    ax = fig.axes[0]
    ax.set_xlim(9.5,11)
    ax.set_ylim(5,6.5)
    #ay = fig.axes[1]
    #ax.set_yticks([ 5, 7])
    plt.show()


In [ ]:
adata_full.obsm['X_umap'][:,0]

In [ ]:
umap_x=adata_full.obsm['X_umap'][:,0]
umap_y=adata_full.obsm['X_umap'][:,1]

adata_insp=adata_full[(umap_x>9.5) & (umap_x<11) & (umap_y>5) & (umap_y<6.5) , :]
adata_insp.obs.target_gene.value_counts()